In [ ]:
import random
from pathlib import Path
from cogent3 import load_aligned_seqs

# Set random seed for reproducibility
random.seed(41)

# Define your input and output directories
fasta_dir = Path("../data/mammals")  # Change to your actual folder path
output_dir = Path("./data")
json_outfile = output_dir / "human_chimp_pairs.json"

output_dir.mkdir(exist_ok=True)
print(f"Output directory: {output_dir.resolve()}")

Output directory: /home/richard/source/BIOL8701-RichardMorris/final_presentation/experiments/4/data


Extract all Human-Chimp pairs from fasta_dir

In [2]:
import json
from cogent3 import make_unaligned_seqs, DNA

def to_taxonomic_name(seqname):
    return seqname.split("-")[0].replace("_", " ")

# Dictionary to store output metadata
alignment_metadata = {}

# Loop through all FASTA files in the directory
for fasta_file in fasta_dir.glob("*.fasta"):
    try:
        # Load aligned sequences
        aln = load_aligned_seqs(fasta_file, moltype=DNA)

        # Rename sequence labels to taxonomic names
        renamed = aln.rename_seqs(lambda name: to_taxonomic_name(name))

        # Subselect human and chimp sequences
        subset = renamed.take_seqs(["homo sapiens", "pan troglodytes"])
        
        # Skip if we don't have both species
        if len(subset.names) != 2:
            continue

        # Ungap and compute differences
        ungapped = subset.degap()
        seq1, seq2 = ungapped.to_dict().values()
        total = sum(a != '-' and b != '-' for a, b in zip(seq1, seq2))
        diff = sum(a != b for a, b in zip(seq1, seq2) if a != '-' and b != '-')

        # Get ensembl id from human sequence name
        ensembl_id = next(name.split("-")[1] for name in aln.names if name.startswith("homo_sapiens-"))

        # Save the ungapped FASTA subset
        out_fasta = output_dir / f"{ensembl_id}.fasta"
        make_unaligned_seqs(subset.degap().to_dict(), moltype=DNA).write(format="fasta", filename=str(out_fasta))

        # Save metadata
        alignment_metadata[ensembl_id] = {
            "filename": out_fasta.name,
            "differences": diff,
            "total": total
        }

    except Exception as e:
        print(f"Skipping {fasta_file.name} due to error: {e}")

# Write metadata to JSON
with open(json_outfile, "w") as f:
    json.dump(alignment_metadata, f, indent=2)

print(f"Processed {len(alignment_metadata)} alignments. Output written to {json_outfile}")


Skipping ENSG00000157881.fasta due to error: 'pan troglodytes'
Skipping ENSG00000160075.fasta due to error: 'pan troglodytes'
Skipping ENSG00000187017.fasta due to error: 'pan troglodytes'
Skipping ENSG00000187583.fasta due to error: 'pan troglodytes'
Skipping ENSG00000187608.fasta due to error: 'pan troglodytes'
Skipping ENSG00000187634.fasta due to error: 'pan troglodytes'
Skipping ENSG00000187961.fasta due to error: 'pan troglodytes'
Skipping ENSG00000188290.fasta due to error: 'pan troglodytes'
Skipping ENSG00000197921.fasta due to error: 'pan troglodytes'
Processed 85 alignments. Output written to data/human_chimp_pairs.json


In [3]:
import numpy as np

# Load JSON metadata
with open(json_outfile) as f:
    alignment_metadata = json.load(f)

# Convert to list of tuples: (ensembl_id, differences, total, prop_diff)
entries = []
for ensembl_id, data in alignment_metadata.items():
    diff = data["differences"]
    total = data["total"]
    if total > 0:  # avoid divide-by-zero
        prop_diff = diff / total
        entries.append((ensembl_id, diff, total, prop_diff))

# Sort entries by proportion difference
entries.sort(key=lambda x: x[3])

# Compute quartile boundaries
num_entries = len(entries)
q1_end = num_entries // 4
q4_start = 3 * num_entries // 4

# Extract 1st and 4th quartile entries
first_quartile = entries[:q1_end]
fourth_quartile = entries[q4_start:]

# Just print how many in each
print(f"Total entries: {num_entries}")
print(f"First quartile (most similar): {len(first_quartile)}")
print(f"Fourth quartile (most divergent): {len(fourth_quartile)}")


Total entries: 85
First quartile (most similar): 21
Fourth quartile (most divergent): 22


In [4]:
import time
import math
import cogent3
from cogent3 import make_aligned_seqs
from cogent3.align import global_pairwise
from cogent3.align.align import make_dna_scoring_dict

from madb import DeBruijnGraph, make_graph


In [ ]:
# Alignment scoring parameters
match = 5
transition = -2
transversion = -4
gap_open = 4
gap_extend = 1

score_matrix = make_dna_scoring_dict(
    match=match, transition=transition, transversion=transversion
)

# Loop through 1st quartile for benchmarking
for ensembl_id, _, _, _ in first_quartile:
    record = alignment_metadata[ensembl_id]
    fasta_path = output_dir / record["filename"]

    try:
        # Load unaligned sequences
        seqs = cogent3.load_unaligned_seqs(fasta_path, moltype=cogent3.DNA)
        s1, s2 = seqs.seqs

        # Time Cogent3 alignment
        start = time.perf_counter()
        cogent3_alignment = global_pairwise(
            s1, s2, score_matrix, gap_open, gap_extend, return_score=False
        )
        cogent3_time = time.perf_counter() - start

        # Time MADB alignment
        dbg = make_graph(seqs, kmer_size=12, moltype=cogent3.DNA)
        start = time.perf_counter()
        madb_alignment = dbg.align(
            match=match, transition=transition, transversion=transversion,
            gap_open=gap_open, gap_extend=gap_extend
        )
        madb_time = time.perf_counter() - start

        # Store results
        record.update({
            "cogent3_time_sec": cogent3_time,
            "madb_time_sec": madb_time
        })

    except Exception as e:
        print(f"[{ensembl_id}] Error during alignment: {e}")
        record.update({
            "error": str(e)
        })

# Write updated results to JSON
with open(json_outfile, "w") as f:
    json.dump(alignment_metadata, f, indent=2)

print("Benchmarking complete. Results saved to", json_outfile)


In [6]:
We should then load the sequences into a madb.DeBruijnGraph with a kmer_size of 12, using madb.make_graph()

def make_graph(data, kmer_size: int = None, moltype : MolType = None)->"DeBruijnGraph":
    """Loads a DeBruijnGraph from a dictionary of sequence data."""
    if moltype is None:
        moltype = cogent3.DNA
    # if data is not a cogent3.SequenceCollection, convert it
    if not isinstance(data, cogent3.SequenceCollection):
        data = cogent3.make_unaligned_seqs(data=data, moltype=moltype)
    if kmer_size is None:
        # find length of longest sequence
        max_seq_len = max(len(seq) for seq in data.seqs) 
        predicted_k = math.log(max_seq_len) - math.log(4) 
        kmer_size = int(predicted_k+1) # round up
        if kmer_size < 3:
            kmer_size = 3  

    dbg = DeBruijnGraph(kmer_length=kmer_size)
    for sequence in data.seqs:
        dbg.add_sequence(sequence)
    return dbg

We should use this method to align the sequences while timing and recording the process

    def align(self, match: int=5, transition: int=-2, transversion: int=-4, gap_open: int=4, gap_extend: int=1):
        # minimize bubbles (using equal length + Karlin)
        bubbles = self.bubbles()
        minimized_bubbles = bubbles.minimize_bubbles()
        minimized_bubbles.align(match, transition, transversion, gap_open, gap_extend)

        result = {}
        for sequence_index in range(1, len(self)+1):
            sequence_name = self.name_for_index(sequence_index) 
            result[sequence_name] = self.alignment(sequence_index, minimized_bubbles)
        return make_aligned_seqs(data=result, moltype=self.moltype) 
 

We should also time and record using cogent3.global_pairwise_align 

And finally we need to rank sequence pairs by pd, select the first and 4th quartile and report mean alignment score and mean time as violin plots


SyntaxError: invalid decimal literal (2053598206.py, line 40)